quarantine and schema drift

In [0]:
%run ./schemas

In [0]:
import json, uuid
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")

BATCH = "qa_quarantine_schema_20261009"
REPO = "apache/spark"
ROOT = Path("/Volumes/github_health/raw/bronze_data/_phase2_qa")
ROOT.mkdir(parents=True, exist_ok=True)
RAW_FILE = ROOT / f"{BATCH}_issues.jsonl"
BQ = "github_health.ops.qa_bronze_quarantine"
SQ = "github_health.ops.qa_silver_quarantine"
DRIFT = "github_health.ops.qa_schema_drift"

spark.sql(f"""CREATE TABLE IF NOT EXISTS {BQ} (
 quarantine_id STRING, batch_id STRING, repository STRING, entity STRING,
 source_file STRING, raw_payload STRING, error_type STRING, error_reason STRING,
 load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
) USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {SQ} (
 quarantine_id STRING, batch_id STRING, repository STRING, entity STRING,
 record_key STRING, raw_payload STRING, error_type STRING, error_reason STRING,
 load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
) USING DELTA""")

# Deterministic test IDs make rerunning assertions reliable.
# Completely synthetic; no real users, commits, or API calls.
base = {
    "number": 15, "title": "QA synthetic issue", "state": "open",
    "created_at": "2026-10-08T11:50:00Z",
    "updated_at": "2026-10-08T11:55:00Z",
    "_repo": REPO, "_batch_id": BATCH
}
examples = [
    {**base, "id": 990000000001},
    {**base, "id": 990000000002, "created_at": "not-a-date"},
    {**base, "number": 16},  # missing issue ID
    {**base, "id": "bad-integer"},  # type mismatch
    '{"id": 990000000005, "title": ',  # malformed JSON
]
RAW_FILE.write_text("\n".join(
    json.dumps(x) if isinstance(x, dict) else x for x in examples
) + "\n", encoding="utf-8")
print("Created isolated test file:", RAW_FILE)
print("Synthetic JSONL lines:", len(examples))
print("Production Bronze and Silver are untouched.")


In [0]:
# Read using the SAME explicit schema and PERMISSIVE mode as production.
raw = (spark.read.schema(RAW_SCHEMAS["issues"])
       .option("mode", "PERMISSIVE")
       .option("columnNameOfCorruptRecord", "_corrupt_record")
       .json(str(RAW_FILE)))

assert raw.count() == 5, "Expected five test lines"
reason = (F.when(F.col("_corrupt_record").isNotNull(), "BAD_JSON_OR_TYPE")
          .when(F.col("id").isNull(), "MISSING_PRIMARY_ID")
          .when(F.col("_repo").isNotNull() & (F.col("_repo") != REPO),
                "REPOSITORY_MISMATCH"))
staged = raw.withColumn("_reason", reason)

# Same quarantine MERGE pattern as common.quarantine, pointed at TEST tables.
def qa_quarantine(source, table):
    source = source.dropDuplicates(["quarantine_id"])
    values = {name: f"s.`{name}`" for name in source.columns}
    (DeltaTable.forName(spark, table).alias("t")
       .merge(source.alias("s"), "t.quarantine_id = s.quarantine_id")
       .whenNotMatchedInsert(values=values).execute())

payload_cols = [F.col(c) for c in RAW_SCHEMAS["issues"].fieldNames()
                if c != "_corrupt_record"]
payload = F.coalesce(F.col("_corrupt_record"), F.to_json(F.struct(*payload_cols)))
failed = staged.filter(F.col("_reason").isNotNull())
bronze_qa = failed.select(
    F.sha2(F.concat_ws("|", F.lit(BATCH), F.lit(REPO), F.lit("issues"), payload), 256).alias("quarantine_id"),
    F.lit(BATCH).alias("batch_id"), F.lit(REPO).alias("repository"),
    F.lit("issues").alias("entity"), F.lit(str(RAW_FILE)).alias("source_file"),
    payload.alias("raw_payload"), F.col("_reason").alias("error_type"),
    F.concat(F.lit("Failed Bronze validation: "), F.col("_reason")).alias("error_reason"),
    F.current_timestamp().alias("load_timestamp"),
    F.current_timestamp().alias("quarantined_at"))
qa_quarantine(bronze_qa, BQ)

valid_bronze = staged.filter(F.col("_reason").isNull())
bad_count = spark.table(BQ).filter(F.col("batch_id") == BATCH).count()
valid_count = valid_bronze.count()
print("Bronze test: quarantined", bad_count, "valid", valid_count)
spark.table(BQ).filter(F.col("batch_id") == BATCH).groupBy("error_type").count().show(truncate=False)
assert bad_count == 3, f"Expected 3 Bronze invalid records, found {bad_count}"
assert valid_count == 2, f"Expected 2 Bronze-valid records, found {valid_count}"
print("BRONZE QUARANTINE TEST PASSED")


In [0]:
# Silver validation follows the driver's required created/updated timestamp rules.
silver_staged = (valid_bronze
    .withColumn("record_key", F.concat_ws("|", F.lit(REPO), F.col("id").cast("string")))
    .withColumn("created_ts", F.expr("try_cast(created_at AS timestamp)"))
    .withColumn("updated_ts", F.expr("try_cast(updated_at AS timestamp)"))
    .withColumn("_silver_reason",
        F.when(F.col("id").isNull(), "NULL_ISSUE_ID")
         .when(F.col("created_ts").isNull() | F.col("updated_ts").isNull(),
               "INVALID_TIMESTAMP")))

silver_failed = silver_staged.filter(F.col("_silver_reason").isNotNull())
silver_payload = F.to_json(F.struct(*payload_cols))
silver_qa = silver_failed.select(
    F.sha2(F.concat_ws("|", F.lit(BATCH), F.lit("issues"),
                       F.col("record_key"), F.col("_silver_reason")), 256).alias("quarantine_id"),
    F.lit(BATCH).alias("batch_id"), F.lit(REPO).alias("repository"),
    F.lit("issues").alias("entity"), F.col("record_key"),
    silver_payload.alias("raw_payload"), F.col("_silver_reason").alias("error_type"),
    F.concat(F.lit("Silver validation failure: "), F.col("_silver_reason")).alias("error_reason"),
    F.current_timestamp().alias("load_timestamp"),
    F.current_timestamp().alias("quarantined_at"))
qa_quarantine(silver_qa, SQ)

silver_bad = spark.table(SQ).filter(F.col("batch_id") == BATCH).count()
silver_good = silver_staged.filter(F.col("_silver_reason").isNull()).count()
print("Silver test: quarantined", silver_bad, "valid", silver_good)
spark.table(SQ).filter(F.col("batch_id") == BATCH).groupBy("error_type").count().show(truncate=False)
assert silver_bad == 1, "Expected one invalid timestamp in Silver quarantine"
assert silver_good == 1, "Expected one good Silver record"
# Rerun MERGE: deterministic IDs ensure no duplicate quarantine rows.
qa_quarantine(bronze_qa, BQ)
qa_quarantine(silver_qa, SQ)
assert spark.table(BQ).filter(F.col("batch_id") == BATCH).count() == 3
assert spark.table(SQ).filter(F.col("batch_id") == BATCH).count() == 1
print("SILVER QUARANTINE + QUARANTINE IDEMPOTENCY TESTS PASSED")


## Schema evolution test

In [0]:
# Only a table prefixed qa_ in ops is replaced. Production Delta tables are untouched.
spark.sql(f"DROP TABLE IF EXISTS {DRIFT}")

TS = datetime(2026, 10, 8, 12, 0, tzinfo=timezone.utc)
BASE_SCHEMA = StructType([
    StructField("record_key", StringType(), False),
    StructField("repository", StringType(), False),
    StructField("title", StringType(), True),
    StructField("payload_hash", StringType(), True),
    StructField("load_timestamp", TimestampType(), True)
])
EXTENDED_SCHEMA = StructType(BASE_SCHEMA.fields + [
    StructField("review_priority", LongType(), True)
])
MERGE_SCHEMA = StructType(EXTENDED_SCHEMA.fields + [
    StructField("triage_tag", StringType(), True)
])

base_df = spark.createDataFrame(
    [("qa_issue_1", REPO, "Original", "hash1", TS)], schema=BASE_SCHEMA)
base_df.write.format("delta").mode("overwrite").saveAsTable(DRIFT)

# 1) Valid drift during APPEND, using per-write mergeSchema.
new_df = spark.createDataFrame(
    [("qa_issue_2", REPO, "Second", "hash2", TS, 3)],
    schema=EXTENDED_SCHEMA)
(new_df.write.format("delta")
 .option("mergeSchema", "true")
 .mode("append").saveAsTable(DRIFT))
assert "review_priority" in spark.table(DRIFT).columns

# 2) Valid drift during an idempotent MERGE using SQL schema evolution.
changes = spark.createDataFrame([
    ("qa_issue_1", REPO, "Updated", "hash3", TS, 1, "review"),
    ("qa_issue_3", REPO, "New", "hash4", TS, 2, "urgent")
], schema=MERGE_SCHEMA)
changes.createOrReplaceTempView("qa_schema_changes_source")
spark.sql(f"""
MERGE WITH SCHEMA EVOLUTION INTO {DRIFT} AS t
USING qa_schema_changes_source AS s
ON t.record_key = s.record_key
WHEN MATCHED AND NOT (t.payload_hash <=> s.payload_hash)
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
assert "triage_tag" in spark.table(DRIFT).columns
assert spark.table(DRIFT).count() == 3
assert spark.table(DRIFT).filter(
    F.col("record_key") == "qa_issue_1").select("title").first()[0] == "Updated"

# Same MERGE repeated: no new rows, original timestamp unchanged.
before = {
    r["record_key"]: r["load_timestamp"]
    for r in spark.table(DRIFT).select("record_key", "load_timestamp").collect()
}
spark.sql(f"""
MERGE WITH SCHEMA EVOLUTION INTO {DRIFT} AS t
USING qa_schema_changes_source AS s
ON t.record_key = s.record_key
WHEN MATCHED AND NOT (t.payload_hash <=> s.payload_hash)
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
after = {
    r["record_key"]: r["load_timestamp"]
    for r in spark.table(DRIFT).select("record_key", "load_timestamp").collect()
}
assert after == before
assert spark.table(DRIFT).count() == 3
spark.table(DRIFT).orderBy("record_key").show(truncate=False)
print("SCHEMA DRIFT (APPEND + MERGE) AND RERUN TESTS PASSED")


In [0]:
%sql

SELECT 'Bronze QA' AS test,
       COUNT(*) AS records
FROM github_health.ops.qa_bronze_quarantine

UNION ALL

SELECT 'Silver QA',
       COUNT(*)
FROM github_health.ops.qa_silver_quarantine

UNION ALL

SELECT 'Schema Drift QA',
       COUNT(*)
FROM github_health.ops.qa_schema_drift;


In [0]:
%sql

DESCRIBE TABLE github_health.ops.qa_schema_drift;
